In [0]:
-- TODO: manual review required — choose a BTEQ replacement wrapper: Databricks notebook, stored procedure, or Workflows DAG.
-- TODO: manual review required — BTEQ ERRORCODE/QUIT handling must be implemented by the chosen wrapper using automatic failure propagation, SIGNAL, or Workflow branches.

/**************************************************************************\
* PREPARATION                                                              *
* ------------------------------------------------------------------------ *
* Screening of the daily payments against the UK consolidated sanctions    *
* list (loaded by kycref.15 / kycref.60). Screened are the counterparty,   *
* ordering customer, beneficiary customer and merchant names, the bank     *
* identifier codes of counterparty and intermediary bank, and the          *
* counterparty / bank country against comprehensively sanctioned regimes.  *
\**************************************************************************/

-- Converted: Teradata index-statistics collection is omitted because Delta tables do not use indexes.


/**************************************************************************\
* BUILD THE LIST OF NAMES TO BE SCREENED                                   *
* ------------------------------------------------------------------------ *
* Name roles:  CP = counterparty          OR = ordering customer           *
*              BE = beneficiary customer  MR = merchant                    *
*              IB = intermediary bank (BIC only, no name)                  *
* The name normalisation is identical to the one applied to the sanctions  *
* list in kycref.60 - any change has to be made in both places.            *
\**************************************************************************/

DELETE FROM DLYTXN_SCREEN_NM
;

-- Converted: Teradata string normalization and trailing-space comparison semantics use Databricks REPLACE, TRANSLATE, and RTRIM-aware predicates.
INSERT INTO DLYTXN_SCREEN_NM
(
       TXN_REF,
       SUPPLIER_CD,
       NM_ROLE_CD,
       RAW_NM,
       NORM_NM,
       CTRY_CD,
       BIC_CD
)
SELECT dtl.TXN_REF,
       dtl.SUPPLIER_CD,
       'CP',
       dtl.CPTY_NM,
       NULLIF (TRIM (REPLACE (REPLACE (TRANSLATE (UPPER (TRIM (dtl.CPTY_NM)), '.,''-&/()', '        '), '  ', ' '), '  ', ' ')), ''),
       COALESCE (CASE WHEN RTRIM(dtl.CPTY_CTRY) = '' THEN NULL ELSE dtl.CPTY_CTRY END, CASE WHEN RTRIM(dtl.CPTY_BANK_CTRY) = '' THEN NULL ELSE dtl.CPTY_BANK_CTRY END),
       CASE WHEN RTRIM(dtl.CPTY_BIC) = '' THEN NULL ELSE dtl.CPTY_BIC END
FROM
       DLYTXN_LOAD   dtl
WHERE
       (dtl.CPTY_NM IS NOT NULL OR RTRIM(dtl.CPTY_BIC) <> '')
;

-- Converted: Teradata string normalization and trailing-space comparison semantics use Databricks REPLACE, TRANSLATE, and RTRIM-aware predicates.
INSERT INTO DLYTXN_SCREEN_NM
(
       TXN_REF,
       SUPPLIER_CD,
       NM_ROLE_CD,
       RAW_NM,
       NORM_NM,
       CTRY_CD,
       BIC_CD
)
SELECT dtl.TXN_REF,
       dtl.SUPPLIER_CD,
       'OR',
       dtl.ORDERING_CUST_NM,
       NULLIF (TRIM (REPLACE (REPLACE (TRANSLATE (UPPER (TRIM (dtl.ORDERING_CUST_NM)), '.,''-&/()', '        '), '  ', ' '), '  ', ' ')), ''),
       NULL,
       NULL
FROM
       DLYTXN_LOAD   dtl
WHERE
       dtl.ORDERING_CUST_NM IS NOT NULL
;

-- Converted: Teradata string normalization and trailing-space comparison semantics use Databricks REPLACE, TRANSLATE, and RTRIM-aware predicates.
INSERT INTO DLYTXN_SCREEN_NM
(
       TXN_REF,
       SUPPLIER_CD,
       NM_ROLE_CD,
       RAW_NM,
       NORM_NM,
       CTRY_CD,
       BIC_CD
)
SELECT dtl.TXN_REF,
       dtl.SUPPLIER_CD,
       'BE',
       dtl.BENEF_CUST_NM,
       NULLIF (TRIM (REPLACE (REPLACE (TRANSLATE (UPPER (TRIM (dtl.BENEF_CUST_NM)), '.,''-&/()', '        '), '  ', ' '), '  ', ' ')), ''),
       NULL,
       NULL
FROM
       DLYTXN_LOAD   dtl
WHERE
       dtl.BENEF_CUST_NM IS NOT NULL
;

-- Converted: Teradata string normalization and trailing-space comparison semantics use Databricks REPLACE, TRANSLATE, and RTRIM-aware predicates.
INSERT INTO DLYTXN_SCREEN_NM
(
       TXN_REF,
       SUPPLIER_CD,
       NM_ROLE_CD,
       RAW_NM,
       NORM_NM,
       CTRY_CD,
       BIC_CD
)
SELECT dtl.TXN_REF,
       dtl.SUPPLIER_CD,
       'MR',
       dtl.MERCHANT_NM,
       NULLIF (TRIM (REPLACE (REPLACE (TRANSLATE (UPPER (TRIM (dtl.MERCHANT_NM)), '.,''-&/()', '        '), '  ', ' '), '  ', ' ')), ''),
       CASE WHEN RTRIM(dtl.MERCHANT_CTRY) = '' THEN NULL ELSE dtl.MERCHANT_CTRY END,
       NULL
FROM
       DLYTXN_LOAD   dtl
WHERE
       dtl.MERCHANT_NM IS NOT NULL
;

-- Converted: Teradata string normalization and trailing-space comparison semantics use Databricks REPLACE, TRANSLATE, and RTRIM-aware predicates.
INSERT INTO DLYTXN_SCREEN_NM
(
       TXN_REF,
       SUPPLIER_CD,
       NM_ROLE_CD,
       RAW_NM,
       NORM_NM,
       CTRY_CD,
       BIC_CD
)
SELECT dtl.TXN_REF,
       dtl.SUPPLIER_CD,
       'IB',
       NULL,
       NULL,
       SUBSTR (dtl.INTERMED_BIC, 5, 2),
       dtl.INTERMED_BIC
FROM
       DLYTXN_LOAD   dtl
WHERE
       RTRIM(dtl.INTERMED_BIC) <> ''
;

-- Converted: Teradata index statistics are omitted; refresh the deliberately loaded column statistics with ANALYZE TABLE.
ANALYZE TABLE DLYTXN_SCREEN_NM COMPUTE STATISTICS FOR COLUMNS NORM_NM;


/**************************************************************************\
* SPLIT THE NORMALISED NAMES INTO TOKENS                                   *
* ------------------------------------------------------------------------ *
* STRTOK_SPLIT_TO_TABLE accepts a single key column, so the business key   *
* is concatenated and split up again afterwards. Noise words (legal forms, *
* titles, articles) and single character tokens are dropped.               *
\**************************************************************************/

-- TODO: manual review required — confirm bounded VARCHAR width semantics for the temporary staging table.
-- Converted: Teradata VOLATILE TABLE becomes a session-scoped TEMP TABLE; PRIMARY INDEX and ON COMMIT clauses are omitted.
CREATE TEMP TABLE VT_SCREEN_SRC
(
       SCREEN_KEY    VARCHAR(50)    NOT NULL,
       NORM_NM       VARCHAR(140)
)
;

INSERT INTO VT_SCREEN_SRC
SELECT TRIM (TXN_REF) || '|' || SUPPLIER_CD || '|' || NM_ROLE_CD,
       NORM_NM
FROM   DLYTXN_SCREEN_NM
WHERE  NORM_NM IS NOT NULL
;

DELETE FROM DLYTXN_SCREEN_TOKEN
;

-- Converted: STRTOK_SPLIT_TO_TABLE becomes POSEXPLODE over nonempty SPLIT elements; TRY_ELEMENT_AT preserves STRTOK missing-token behavior.
INSERT INTO DLYTXN_SCREEN_TOKEN
(
       TXN_REF,
       SUPPLIER_CD,
       NM_ROLE_CD,
       TOKEN_SEQ_NR,
       TOKEN_TXT
)
SELECT TRY_ELEMENT_AT(FILTER(SPLIT(tok.OUTKEY, '[|]'), x -> LENGTH(x) > 0), 1),
       TRY_ELEMENT_AT(FILTER(SPLIT(tok.OUTKEY, '[|]'), x -> LENGTH(x) > 0), 2),
       TRY_ELEMENT_AT(FILTER(SPLIT(tok.OUTKEY, '[|]'), x -> LENGTH(x) > 0), 3),
       tok.TOKENNUM,
       tok.TOKEN
FROM
       (SELECT src.SCREEN_KEY AS OUTKEY,
               p.pos + 1 AS TOKENNUM,
               p.TOKEN
        FROM VT_SCREEN_SRC src
        LATERAL VIEW POSEXPLODE(FILTER(SPLIT(src.NORM_NM, ' '), x -> LENGTH(x) > 0)) p AS pos, TOKEN
       ) tok

       LEFT OUTER JOIN
       NAME_NOISE_WORD       nsw
          ON  RTRIM(nsw.NOISE_WORD) = RTRIM(tok.TOKEN)
WHERE
       nsw.NOISE_WORD IS NULL
AND    CHARACTER_LENGTH (TRIM (tok.TOKEN)) >= 2
;

-- Converted: refresh the deliberately loaded token-column statistics with Databricks ANALYZE TABLE.
ANALYZE TABLE DLYTXN_SCREEN_TOKEN COMPUTE STATISTICS FOR COLUMNS TOKEN_TXT, TXN_REF, SUPPLIER_CD, NM_ROLE_CD;

-- Converted: drop the converted session temporary table explicitly.
DROP TEMPORARY TABLE VT_SCREEN_SRC
;


/**************************************************************************\
* CANDIDATES 1 - EXACT NAME MATCH                                          *
* ------------------------------------------------------------------------ *
* Normalised screened name equals a normalised name or alias of an         *
* active designated person / entity. Score 100.                            *
\**************************************************************************/

DELETE FROM DLYTXN_SCREEN_CAND
;

-- Converted: character match predicates use RTRIM to preserve Teradata trailing-space equivalence; decimal coverage ratios use explicit precision.
INSERT INTO DLYTXN_SCREEN_CAND
(
       TXN_REF,
       SUPPLIER_CD,
       NM_ROLE_CD,
       GROUP_ID,
       NAME_SEQ_NR,
       MATCH_TOKEN_CNT,
       SANC_TOKEN_CNT,
       SCREEN_TOKEN_CNT,
       MATCH_TYP_CD,
       MATCH_SCORE
)
SELECT scr.TXN_REF,
       scr.SUPPLIER_CD,
       scr.NM_ROLE_CD,
       snm.GROUP_ID,
       snm.NAME_SEQ_NR,
       snm.TOKEN_CNT,
       snm.TOKEN_CNT,
       snm.TOKEN_CNT,
       'EXACT',
       100.00
FROM
       DLYTXN_SCREEN_NM  scr

       INNER JOIN
       SANC_NAME                   snm
          ON  RTRIM(snm.NORM_NM) = RTRIM(scr.NORM_NM)

       INNER JOIN
       SANC_ENTITY                 sen
          ON  sen.GROUP_ID   = snm.GROUP_ID
          AND RTRIM(sen.ACTIVE_IND) = 'Y'
WHERE
       scr.NORM_NM IS NOT NULL
;


/**************************************************************************\
* CANDIDATES 2 - TOKEN MATCH                                               *
* ------------------------------------------------------------------------ *
* Fuzzy match on name tokens: at least two distinct tokens of the screened *
* name are found in one listed name, and they cover at least 75 percent    *
* of the tokens of the listed name. This catches re-ordered names          *
* (SURNAME FORENAME), additional middle names and added legal forms.       *
* Score = covered percentage, capped at 99 to rank below exact matches.    *
* Low quality aliases (ALIAS_QUALITY_CD = L) need full coverage.           *
\**************************************************************************/

-- Converted: character match predicates use RTRIM to preserve Teradata trailing-space equivalence; decimal coverage ratios use explicit precision.
INSERT INTO DLYTXN_SCREEN_CAND
(
       TXN_REF,
       SUPPLIER_CD,
       NM_ROLE_CD,
       GROUP_ID,
       NAME_SEQ_NR,
       MATCH_TOKEN_CNT,
       SANC_TOKEN_CNT,
       SCREEN_TOKEN_CNT,
       MATCH_TYP_CD,
       MATCH_SCORE
)
SELECT mtc.TXN_REF,
       mtc.SUPPLIER_CD,
       mtc.NM_ROLE_CD,
       mtc.GROUP_ID,
       mtc.NAME_SEQ_NR,
       mtc.MATCH_TOKEN_CNT,
       snm.TOKEN_CNT,
       stc.SCREEN_TOKEN_CNT,
       'TOKEN',
       CASE
          WHEN CAST(mtc.MATCH_TOKEN_CNT * 100.00 / snm.TOKEN_CNT AS DECIMAL(18,6)) >= 99.00
          THEN 99.00
          ELSE CAST (mtc.MATCH_TOKEN_CNT * 100.00 / snm.TOKEN_CNT AS DECIMAL(5,2))
       END
FROM
       (SELECT tkn.TXN_REF,
               tkn.SUPPLIER_CD,
               tkn.NM_ROLE_CD,
               snt.GROUP_ID,
               snt.NAME_SEQ_NR,
               COUNT (DISTINCT tkn.TOKEN_TXT)     AS MATCH_TOKEN_CNT
        FROM   DLYTXN_SCREEN_TOKEN   tkn
               INNER JOIN
               SANC_NAME_TOKEN                 snt
                  ON  RTRIM(snt.TOKEN_TXT) = RTRIM(tkn.TOKEN_TXT)
        GROUP  BY 1, 2, 3, 4, 5
        HAVING COUNT (DISTINCT tkn.TOKEN_TXT) >= 2
       )                                          mtc

       INNER JOIN
       SANC_NAME                   snm
          ON  snm.GROUP_ID    = mtc.GROUP_ID
          AND snm.NAME_SEQ_NR = mtc.NAME_SEQ_NR

       INNER JOIN
       SANC_ENTITY                 sen
          ON  sen.GROUP_ID   = mtc.GROUP_ID
          AND RTRIM(sen.ACTIVE_IND) = 'Y'

       INNER JOIN
       (SELECT TXN_REF,
               SUPPLIER_CD,
               NM_ROLE_CD,
               COUNT (DISTINCT TOKEN_TXT)         AS SCREEN_TOKEN_CNT
        FROM   DLYTXN_SCREEN_TOKEN
        GROUP  BY 1, 2, 3
       )                                          stc
          ON  stc.TXN_REF     = mtc.TXN_REF
          AND stc.SUPPLIER_CD = mtc.SUPPLIER_CD
          AND stc.NM_ROLE_CD  = mtc.NM_ROLE_CD
WHERE
       snm.TOKEN_CNT > 0
AND    CAST(mtc.MATCH_TOKEN_CNT * 1.00 / snm.TOKEN_CNT AS DECIMAL(18,6)) >= 0.75
AND    (   RTRIM(COALESCE (snm.ALIAS_QUALITY_CD, 'G')) <> 'L'
        OR mtc.MATCH_TOKEN_CNT >= snm.TOKEN_CNT)
;


/**************************************************************************\
* CANDIDATES 3 - SANCTIONED COUNTRY                                        *
* ------------------------------------------------------------------------ *
* Counterparty country or counterparty bank country belongs to a regime    *
* with comprehensive (country wide) sanctions. Not tied to a designated    *
* person, hence GROUP_ID = -1 and NAME_SEQ_NR = 0. Score 80.               *
\**************************************************************************/

-- Converted: character match predicates use RTRIM to preserve Teradata trailing-space equivalence; decimal coverage ratios use explicit precision.
INSERT INTO DLYTXN_SCREEN_CAND
(
       TXN_REF,
       SUPPLIER_CD,
       NM_ROLE_CD,
       GROUP_ID,
       NAME_SEQ_NR,
       MATCH_TOKEN_CNT,
       SANC_TOKEN_CNT,
       SCREEN_TOKEN_CNT,
       MATCH_TYP_CD,
       MATCH_SCORE
)
SELECT DISTINCT
       cpt.TXN_REF,
       cpt.SUPPLIER_CD,
       'CP',
       -1,
       0,
       0,
       0,
       0,
       'CTRY',
       80.00
FROM
       DLYTXN_CPTY       cpt

       INNER JOIN
       SANC_REGIME_CTRY      rgc
          ON  RTRIM(rgc.CTRY_2_CD) IN (RTRIM(cpt.CPTY_CTRY_CD), RTRIM(cpt.CPTY_BANK_CTRY_CD))
          AND RTRIM(rgc.COMPREHENSIVE_IND) = 'Y'
;

-- Converted: character match predicates use RTRIM to preserve Teradata trailing-space equivalence; decimal coverage ratios use explicit precision.
INSERT INTO DLYTXN_SCREEN_CAND
(
       TXN_REF,
       SUPPLIER_CD,
       NM_ROLE_CD,
       GROUP_ID,
       NAME_SEQ_NR,
       MATCH_TOKEN_CNT,
       SANC_TOKEN_CNT,
       SCREEN_TOKEN_CNT,
       MATCH_TYP_CD,
       MATCH_SCORE
)
SELECT DISTINCT
       scr.TXN_REF,
       scr.SUPPLIER_CD,
       scr.NM_ROLE_CD,
       -1,
       0,
       0,
       0,
       0,
       'CTRY',
       80.00
FROM
       DLYTXN_SCREEN_NM  scr

       INNER JOIN
       SANC_REGIME_CTRY      rgc
          ON  RTRIM(rgc.CTRY_2_CD) = RTRIM(scr.CTRY_CD)
          AND RTRIM(rgc.COMPREHENSIVE_IND) = 'Y'
WHERE
       RTRIM(scr.NM_ROLE_CD) IN ('MR', 'IB')
;


/**************************************************************************\
* CANDIDATES 4 - DESIGNATED BANK IDENTIFIER CODE                           *
* ------------------------------------------------------------------------ *
* BIC of counterparty bank or intermediary bank belongs to a designated    *
* entity. Compared on the 8 character institution / location part as       *
* well, as branch codes are often omitted. Score 100.                      *
\**************************************************************************/

-- Converted: character match predicates use RTRIM to preserve Teradata trailing-space equivalence; decimal coverage ratios use explicit precision.
INSERT INTO DLYTXN_SCREEN_CAND
(
       TXN_REF,
       SUPPLIER_CD,
       NM_ROLE_CD,
       GROUP_ID,
       NAME_SEQ_NR,
       MATCH_TOKEN_CNT,
       SANC_TOKEN_CNT,
       SCREEN_TOKEN_CNT,
       MATCH_TYP_CD,
       MATCH_SCORE
)
SELECT scr.TXN_REF,
       scr.SUPPLIER_CD,
       scr.NM_ROLE_CD,
       sbc.GROUP_ID,
       0,
       0,
       0,
       0,
       'BIC',
       CASE WHEN RTRIM(sbc.BIC_CD) = RTRIM(scr.BIC_CD) THEN 100.00 ELSE 95.00 END
FROM
       DLYTXN_SCREEN_NM  scr

       INNER JOIN
       SANC_BIC                    sbc
          ON  RTRIM(SUBSTR (sbc.BIC_CD, 1, 8)) = RTRIM(SUBSTR (scr.BIC_CD, 1, 8))

       INNER JOIN
       SANC_ENTITY                 sen
          ON  sen.GROUP_ID   = sbc.GROUP_ID
          AND RTRIM(sen.ACTIVE_IND) = 'Y'
WHERE
       scr.BIC_CD IS NOT NULL
;

-- Converted: refresh the deliberately loaded candidate-column statistics with Databricks ANALYZE TABLE.
ANALYZE TABLE DLYTXN_SCREEN_CAND COMPUTE STATISTICS FOR COLUMNS TXN_REF, SUPPLIER_CD, NM_ROLE_CD, GROUP_ID;


/**************************************************************************\
* WRITE NEW SCREENING HITS                                                 *
* ------------------------------------------------------------------------ *
* Best candidate per transaction, name role and listed entity. Hits which  *
* were already raised for the same transaction (re-delivery, status        *
* update) are not raised again so that the screening analysts do not see   *
* duplicates in their queue. HIT_ID is an identity column.                 *
\**************************************************************************/

-- Converted: hit-selection character predicates and window keys normalize trailing spaces to preserve Teradata comparison and partition behavior.
INSERT INTO SANC_SCREEN_HIT
(
       TXN_REF,
       SUPPLIER_CD,
       BUSINESS_DT,
       ACCT_ID,
       PARTY_ID,
       NM_ROLE_CD,
       SCREENED_NM,
       GROUP_ID,
       MATCHED_NM,
       REGIME_NM,
       MATCH_TYP_CD,
       MATCH_SCORE,
       HIT_STATUS_CD,
       CO_UPD_TMS
)
SELECT cnd.TXN_REF,
       cnd.SUPPLIER_CD,
       tms.BUSINESS_DT,
       dtl.ACCT_ID,
       acs.PRIMARY_PARTY_ID,
       cnd.NM_ROLE_CD,
       CASE
          WHEN RTRIM(cnd.MATCH_TYP_CD) = 'BIC'  THEN scr.BIC_CD
          WHEN RTRIM(cnd.MATCH_TYP_CD) = 'CTRY' AND RTRIM(cnd.NM_ROLE_CD) = 'CP'
          THEN COALESCE (scr.RAW_NM, cpt.CPTY_ACCT_KEY)
          ELSE scr.RAW_NM
       END,
       cnd.GROUP_ID,
       CASE
          WHEN RTRIM(cnd.MATCH_TYP_CD) = 'CTRY'
          THEN 'COUNTRY ' || COALESCE (rgc.CTRY_2_CD, '')
          WHEN RTRIM(cnd.MATCH_TYP_CD) = 'BIC'
          THEN 'BIC OF GROUP ' || TRIM (CAST (cnd.GROUP_ID AS STRING))
          ELSE snm.FULL_NM
       END,
       COALESCE (sen.REGIME_NM, rgc.REGIME_NM),
       cnd.MATCH_TYP_CD,
       cnd.MATCH_SCORE,
       'OPEN',
       tms.LOAD_TMS
FROM
       DLYTXN_SCREEN_CAND   cnd

       CROSS JOIN
       DLYTXN_LOAD_TMS      tms

       INNER JOIN
       DLYTXN_LOAD          dtl
          ON  RTRIM(dtl.TXN_REF)     = RTRIM(cnd.TXN_REF)
          AND RTRIM(dtl.SUPPLIER_CD) = RTRIM(cnd.SUPPLIER_CD)

       LEFT OUTER JOIN
       DLYTXN_SCREEN_NM     scr
          ON  RTRIM(scr.TXN_REF)     = RTRIM(cnd.TXN_REF)
          AND RTRIM(scr.SUPPLIER_CD) = RTRIM(cnd.SUPPLIER_CD)
          AND RTRIM(scr.NM_ROLE_CD)  = RTRIM(cnd.NM_ROLE_CD)

       LEFT OUTER JOIN
       DLYTXN_CPTY          cpt
          ON  RTRIM(cpt.TXN_REF)     = RTRIM(cnd.TXN_REF)
          AND RTRIM(cpt.SUPPLIER_CD) = RTRIM(cnd.SUPPLIER_CD)

       LEFT OUTER JOIN
       ACCT_SNAPSHOT                  acs
          ON  acs.ACCT_ID = dtl.ACCT_ID

       LEFT OUTER JOIN
       SANC_NAME                      snm
          ON  snm.GROUP_ID    = cnd.GROUP_ID
          AND snm.NAME_SEQ_NR = cnd.NAME_SEQ_NR

       LEFT OUTER JOIN
       SANC_ENTITY                    sen
          ON  sen.GROUP_ID = cnd.GROUP_ID

       LEFT OUTER JOIN
       SANC_REGIME_CTRY         rgc
          ON  RTRIM(cnd.MATCH_TYP_CD)      = 'CTRY'
          AND RTRIM(rgc.COMPREHENSIVE_IND) = 'Y'
          AND RTRIM(rgc.CTRY_2_CD) IN (RTRIM(COALESCE (cpt.CPTY_CTRY_CD, scr.CTRY_CD)),
                                       RTRIM(COALESCE (cpt.CPTY_BANK_CTRY_CD, scr.CTRY_CD)))
WHERE
       NOT EXISTS
       (SELECT 1
        FROM   SANC_SCREEN_HIT   hit
        WHERE  RTRIM(hit.TXN_REF)     = RTRIM(cnd.TXN_REF)
        AND    RTRIM(hit.SUPPLIER_CD) = RTRIM(cnd.SUPPLIER_CD)
        AND    hit.GROUP_ID             = cnd.GROUP_ID
        AND    RTRIM(hit.NM_ROLE_CD)    = RTRIM(cnd.NM_ROLE_CD)
       )

QUALIFY ROW_NUMBER () OVER (PARTITION BY RTRIM(cnd.TXN_REF),
                                         RTRIM(cnd.SUPPLIER_CD),
                                         RTRIM(cnd.NM_ROLE_CD),
                                         cnd.GROUP_ID
                                ORDER BY cnd.MATCH_SCORE DESC,
                                         RTRIM(cnd.MATCH_TYP_CD) ASC,
                                         cnd.NAME_SEQ_NR ASC,
                                         RTRIM(rgc.REGIME_NM) ASC
                           ) = 1
;

-- Converted: Teradata index statistics are omitted; refresh the deliberately loaded business-date statistics with ANALYZE TABLE.
ANALYZE TABLE SANC_SCREEN_HIT COMPUTE STATISTICS FOR COLUMNS BUSINESS_DT;


/**************************************************************************\
* SCREENING SUMMARY FOR THE BATCH LOG                                      *
\**************************************************************************/

-- Converted: use a trailing-space-insensitive collation on summary group keys to preserve Teradata grouping semantics.
SELECT hit.MATCH_TYP_CD COLLATE UTF8_BINARY_RTRIM AS MATCH_TYP_CD,
       hit.NM_ROLE_CD COLLATE UTF8_BINARY_RTRIM AS NM_ROLE_CD,
       COUNT (*)                      AS HIT_CNT,
       COUNT (DISTINCT hit.TXN_REF COLLATE UTF8_BINARY_RTRIM) AS TXN_CNT
FROM
       SANC_SCREEN_HIT             hit

       INNER JOIN
       DLYTXN_LOAD_TMS   tms
          ON  tms.LOAD_TMS = hit.CO_UPD_TMS
GROUP  BY 1, 2
ORDER  BY 1, 2
;


